# Notebook 3 - Estrategia Momentum y Senales

Este notebook implementa la senal de momentum tipo MSCI Momentum con rebalanceo mensual en el ultimo dia habil (ultimo dato disponible del mes).

Objetivo:
- Construir senales de **R_12** y **R_6** con retornos logaritmicos y lag de 1 mes.
- Estandarizar cross-sectional por mes (Z_12 y Z_6).
- Construir score final y seleccionar TOP 20 activos por fecha de rebalanceo.
- Exportar el resultado a CSV para Notebook 4.


## Reglas del modelo (sin look-ahead)

Para cada fecha de rebalanceo mensual `t`:
- `R_12[t] = log(Pm[t-1] / Pm[t-13])`
- `R_6[t] = log(Pm[t-1] / Pm[t-7])`

Esto excluye el mes actual (`shift(1)`) y evita sesgo de look-ahead.

Normalizacion mensual cross-sectional:
- `Z_12[t, i] = (R_12[t, i] - mu_t) / sigma_t`
- `Z_6[t, i] = (R_6[t, i] - mu_t) / sigma_t`

Decision para `sigma_t = 0`:
- Se asigna `Z = 0` a los activos no nulos de esa fila.


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as fs

sns.set(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 200)

# Configuracion temporal de estrategia/senal
BACKTEST_START = pd.Timestamp("2015-01-01")
WARMUP_MONTHS = 13
WARMUP_START = BACKTEST_START - pd.DateOffset(months=WARMUP_MONTHS)
TOP_N = 20

# Filtros anti-outliers para limpiar seleccion final
MIN_DAILY_OBS_PRE = 180
MIN_LAST_PRICE_PRE = 5.0
MAX_P99_ABS_RET_PRE = 0.20
MAX_MAX_ABS_RET_PRE = 1.00
MAX_ANNUAL_VOL_PRE = 1.20

print("BACKTEST_START:", BACKTEST_START.date())
print("WARMUP_START (13 meses antes):", WARMUP_START.date())
print(
    "Filtros outliers ->",
    f"obs>={MIN_DAILY_OBS_PRE}, precio>={MIN_LAST_PRICE_PRE},",
    f"p99_abs_ret<={MAX_P99_ABS_RET_PRE}, max_abs_ret<={MAX_MAX_ABS_RET_PRE}, vol_anual<={MAX_ANNUAL_VOL_PRE}"
)


BACKTEST_START: 2015-01-01
WARMUP_START (13 meses antes): 2013-12-01
Filtros outliers -> obs>=180, precio>=5.0, p99_abs_ret<=0.2, max_abs_ret<=1.0, vol_anual<=1.2


## Funciones reutilizables (para Notebook 4)

Se definen funciones pequenas para reutilizar el pipeline de senales.


In [2]:
def to_wide_close(df):
    """
    Convierte input a matriz wide de precios close:
    index = fecha, columns = tickers.
    Acepta:
    - formato long con [date, ticker/symbol, close/adj_close]
    - formato wide con DatetimeIndex
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise ValueError("Input vacio o no es DataFrame.")

    x = df.copy()
    x.columns = [str(c).strip().lower() for c in x.columns]

    date_cands = ["date", "datetime", "timestamp", "fecha"]
    ticker_cands = ["ticker", "symbol", "asset", "activo"]
    close_cands = ["adj_close", "close", "unadjusted_close", "price"]

    date_col = next((c for c in date_cands if c in x.columns), None)
    ticker_col = next((c for c in ticker_cands if c in x.columns), None)
    close_col = next((c for c in close_cands if c in x.columns), None)

    # Caso long: date + ticker + close
    if date_col is not None and ticker_col is not None and close_col is not None:
        y = x[[date_col, ticker_col, close_col]].copy()
        y[date_col] = pd.to_datetime(y[date_col], errors="coerce")
        y[ticker_col] = y[ticker_col].astype(str).str.upper()
        y = y.dropna(subset=[date_col, ticker_col, close_col])
        px = y.pivot_table(index=date_col, columns=ticker_col, values=close_col, aggfunc="last")
        px = px.sort_index()
        return px

    # Caso wide con datetime index
    y = df.copy()
    if not isinstance(y.index, pd.DatetimeIndex):
        if date_col is None:
            raise ValueError("No encuentro columna fecha ni DatetimeIndex para construir matriz de precios.")
        y[date_col] = pd.to_datetime(y[date_col], errors="coerce")
        y = y.dropna(subset=[date_col]).set_index(date_col)

    for c in y.columns:
        y[c] = pd.to_numeric(y[c], errors="coerce")

    y = y.select_dtypes(include=[np.number]).sort_index()
    y.columns = [str(c).strip().upper() for c in y.columns]
    y = y.dropna(axis=1, how="all")
    return y


def to_monthly_last(px_close):
    """
    Convierte precios a mensual usando ultimo dato disponible de cada mes.
    - Si parece diario/intradiario: resample BM + last.
    - Si ya parece mensual: colapsa por mes y toma ultimo disponible.
    """
    if not isinstance(px_close.index, pd.DatetimeIndex):
        raise TypeError("px_close debe tener DatetimeIndex.")

    px = px_close.sort_index().copy()
    px = px[~px.index.duplicated(keep="last")]
    px = px.dropna(axis=1, how="all")

    obs_per_month = px.groupby(px.index.to_period("M")).size()
    is_daily_like = obs_per_month.median() > 1

    if is_daily_like:
        Pm = px.resample("BM").last()
    else:
        Pm = px.groupby(px.index.to_period("M")).last()
        Pm.index = Pm.index.to_timestamp("M")

    Pm = Pm.sort_index().dropna(axis=1, how="all")
    return Pm


def compute_momentum(Pm):
    """
    Senales momentum con lag 1 mes (sin look-ahead):
    R12[t] = log(Pm[t-1]/Pm[t-13])
    R6[t]  = log(Pm[t-1]/Pm[t-7])
    """
    R12 = np.log(Pm.shift(1) / Pm.shift(13))
    R6 = np.log(Pm.shift(1) / Pm.shift(7))
    return R12, R6


def cross_sectional_zscore(df):
    """
    Z-score cross-sectional por fecha (fila), ignorando NaN.
    Regla std=0: Z=0 para valores no nulos de esa fila.
    """
    mu = df.mean(axis=1, skipna=True)
    sigma = df.std(axis=1, ddof=0, skipna=True)

    z = df.sub(mu, axis=0).div(sigma.replace(0, np.nan), axis=0)

    zero_std_idx = sigma[sigma == 0].index
    if len(zero_std_idx) > 0:
        z.loc[zero_std_idx] = np.where(df.loc[zero_std_idx].notna(), 0.0, np.nan)

    return z


def build_scores_and_select(R12, R6, top_n=20, hold_last_weights=True):
    """
    Construye score, rankea y selecciona top_n por fecha.
    - Sin GLD (se asume GLD ya fuera de columnas).
    - Empates: score desc, ticker asc (estable y reproducible).
    - Si en una fecha hay <top_n scores validos:
        * hold_last_weights=True: mantiene pesos del mes anterior.
        * si no hay cartera previa: se queda en cash.
    Devuelve:
    - selection_wide: matriz fecha x activo con pesos (0 o 1/top_n).
    - selection_long: tabla fecha-activo-score-rank (+ z12,z6,r12,r6,weight,rebalanced).
    """
    if R12.shape != R6.shape:
        raise ValueError("R12 y R6 deben tener misma forma.")

    z12 = cross_sectional_zscore(R12)
    z6 = cross_sectional_zscore(R6)
    score = 0.5 * (z12 + z6)

    valid_score_count = score.notna().sum(axis=1)
    first_rebalance = valid_score_count[valid_score_count >= top_n].index.min()
    if pd.isna(first_rebalance):
        raise ValueError(f"No hay ninguna fecha con al menos {top_n} scores validos.")

    tickers = score.columns.tolist()
    dates = score.index.tolist()

    selection_wide = pd.DataFrame(0.0, index=dates, columns=tickers)
    long_rows = []

    prev_weights = pd.Series(0.0, index=tickers)
    prev_ranked = []
    invested = False

    for dt in dates:
        row = pd.DataFrame({
            "ticker": tickers,
            "score": score.loc[dt].values,
            "z12": z12.loc[dt].values,
            "z6": z6.loc[dt].values,
            "r12": R12.loc[dt].values,
            "r6": R6.loc[dt].values,
        }).dropna(subset=["score"])

        has_enough = len(row) >= top_n

        if has_enough:
            ranked = row.sort_values(["score", "ticker"], ascending=[False, True]).head(top_n).reset_index(drop=True)
            ranked["rank"] = np.arange(1, top_n + 1)

            current_weights = pd.Series(0.0, index=tickers)
            current_weights.loc[ranked["ticker"].tolist()] = 1.0 / top_n

            prev_weights = current_weights.copy()
            prev_ranked = ranked["ticker"].tolist()
            invested = True
            rebalanced_flag = 1
        else:
            if hold_last_weights and invested:
                current_weights = prev_weights.copy()
                ranked = pd.DataFrame({
                    "ticker": prev_ranked,
                    "rank": np.arange(1, len(prev_ranked) + 1),
                })

                map_score = score.loc[dt].to_dict()
                map_z12 = z12.loc[dt].to_dict()
                map_z6 = z6.loc[dt].to_dict()
                map_r12 = R12.loc[dt].to_dict()
                map_r6 = R6.loc[dt].to_dict()

                ranked["score"] = ranked["ticker"].map(map_score)
                ranked["z12"] = ranked["ticker"].map(map_z12)
                ranked["z6"] = ranked["ticker"].map(map_z6)
                ranked["r12"] = ranked["ticker"].map(map_r12)
                ranked["r6"] = ranked["ticker"].map(map_r6)
                rebalanced_flag = 0
            else:
                current_weights = pd.Series(0.0, index=tickers)
                ranked = pd.DataFrame(columns=["ticker", "rank", "score", "z12", "z6", "r12", "r6"])
                rebalanced_flag = 0

        selection_wide.loc[dt] = current_weights.values

        if invested and len(ranked) > 0:
            if "score" not in ranked.columns:
                ranked = ranked.merge(row, on="ticker", how="left")

            for _, r in ranked.iterrows():
                tkr = r["ticker"]
                long_rows.append({
                    "rebalance_date": dt,
                    "ticker": tkr,
                    "rank": int(r["rank"]),
                    "score": r.get("score", np.nan),
                    "z12": r.get("z12", np.nan),
                    "z6": r.get("z6", np.nan),
                    "r12": r.get("r12", np.nan),
                    "r6": r.get("r6", np.nan),
                    "weight": float(current_weights.get(tkr, 0.0)),
                    "rebalanced": rebalanced_flag,
                })

    selection_long = pd.DataFrame(long_rows).sort_values(["rebalance_date", "rank", "ticker"]).reset_index(drop=True)
    return selection_wide, selection_long


## Carga de datos desde NB1/NB2 (sin redescarga)

Se intenta leer primero variables en memoria (NB2) y luego ficheros de `data/`, `output/` y `outputs/` (parquet/csv).
No se anade GLD; el universo se trabaja sin GLD.


In [3]:
def _is_valid_file(path, min_size=100):
    local_fs = fs.LocalFileSystem()
    try:
        info = local_fs.get_file_info(path)
        if info.type != fs.FileType.File:
            return False
        if info.size is None:
            return False
        return info.size > min_size
    except Exception:
        return False


def _scan_candidate_files():
    local_fs = fs.LocalFileSystem()

    roots = [
        "data", "output", "outputs", ".",
        "..", "../data", "../output", "../outputs",
        r"C:\\Users\\alons\\Desktop",
    ]

    found = []
    seen = set()

    # Rutas directas tipicas del proyecto/curso
    fixed_candidates = [
        r"C:\\Users\\alons\\Desktop\\Práctica 7\\sp500_history.parquet",
        r"C:\\Users\\alons\\Desktop\\Practica 7\\sp500_history.parquet",
        r"data\\raw\\sp500_history.parquet",
        r"..\\data\\raw\\sp500_history.parquet",
    ]

    for path in fixed_candidates:
        if path not in seen and _is_valid_file(path):
            info = local_fs.get_file_info(path)
            found.append((path, info.size))
            seen.add(path)

    for root in roots:
        try:
            infos = local_fs.get_file_info(fs.FileSelector(root, recursive=True))
        except Exception:
            continue

        for info in infos:
            if info.type != fs.FileType.File:
                continue

            path = info.path
            path_l = path.lower()

            if not (path_l.endswith(".parquet") or path_l.endswith(".csv")):
                continue

            if "notebook_3_estrategia_momentum_senales" in path_l:
                continue
            if "selected_top20_by_rebalance" in path_l:
                continue

            if info.size is None or info.size <= 100:
                continue

            if path not in seen:
                found.append((path, info.size))
                seen.add(path)

    def _score(path, size):
        p = path.lower()
        s = 0
        if "processed" in p or "procesado" in p or "prepared" in p or "prepar" in p:
            s += 8
        if "sp500" in p or "history" in p:
            s += 10
        if "data" in p:
            s += 2
        if p.endswith(".parquet"):
            s += 3
        if size is not None and size > 1024:
            s += 2
        return s

    found = sorted(found, key=lambda x: (_score(x[0], x[1]), x[1]), reverse=True)
    return [x[0] for x in found]


def _read_any_table(path):
    path_l = path.lower()
    try:
        if path_l.endswith(".parquet"):
            return pq.read_table(path).to_pandas()
        if path_l.endswith(".csv"):
            return pd.read_csv(path)
    except Exception:
        return None
    return None


def _to_close_wide(df):
    if df is None or len(df) == 0:
        return None

    work = df.copy()
    work.columns = [str(c).strip().lower() for c in work.columns]

    date_cands = ["date", "datetime", "timestamp", "fecha"]
    ticker_cands = ["symbol", "ticker", "asset", "activo"]
    close_cands = ["adj_close", "close", "unadjusted_close", "price"]

    date_col = next((c for c in date_cands if c in work.columns), None)
    ticker_col = next((c for c in ticker_cands if c in work.columns), None)
    close_col = next((c for c in close_cands if c in work.columns), None)

    wide = None

    if date_col is not None and ticker_col is not None and close_col is not None:
        tmp = work[[date_col, ticker_col, close_col]].copy()
        tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
        tmp[ticker_col] = tmp[ticker_col].astype(str).str.upper()
        tmp = tmp.dropna(subset=[date_col, ticker_col, close_col])
        wide = tmp.pivot_table(index=date_col, columns=ticker_col, values=close_col, aggfunc="last")

    elif date_col is not None:
        tmp = work.copy()
        tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
        tmp = tmp.dropna(subset=[date_col]).set_index(date_col)

        for c in tmp.columns:
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

        num_cols = tmp.select_dtypes(include=[np.number]).columns.tolist()
        if len(num_cols) >= 2:
            wide = tmp[num_cols].copy()
            wide.columns = [str(c).strip().upper() for c in wide.columns]

    elif isinstance(df.index, pd.DatetimeIndex):
        tmp = df.copy()
        for c in tmp.columns:
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
        num_cols = tmp.select_dtypes(include=[np.number]).columns.tolist()
        if len(num_cols) >= 2:
            wide = tmp[num_cols].copy()
            wide.columns = [str(c).strip().upper() for c in wide.columns]

    if wide is None or wide.shape[1] == 0:
        return None

    wide = wide.replace([np.inf, -np.inf], np.nan)
    wide.index = pd.to_datetime(wide.index, errors="coerce")
    wide = wide[~wide.index.isna()]
    wide = wide.sort_index()
    wide = wide[~wide.index.duplicated(keep="last")]
    wide = wide.dropna(axis=1, how="all")

    if wide.shape[1] == 0:
        return None

    return wide


def _load_from_nb2_globals():
    candidate_names = [
        "datos_analisis_df",
        "df_prepared",
        "datos_base_df",
        "datos_auditoria_df",
        "df",
        "data",
    ]

    for name in candidate_names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            px = _to_close_wide(globals()[name])
            if px is not None and px.shape[1] > 0:
                return px, f"Notebook2::{name}"

    return None, None


def load_prepared_close_prices(min_assets=20):
    # 1) Primero: datos en memoria de NB2
    px_mem, mem_source = _load_from_nb2_globals()
    if px_mem is not None:
        if px_mem.shape[1] >= min_assets and px_mem.shape[0] >= 260:
            print(f"Fuente seleccionada: {mem_source}")
            return px_mem, mem_source
        print(f"Fuente en memoria detectada ({mem_source}) pero con cobertura baja: {px_mem.shape}")

    # 2) PARQUET_PATH si viene de NB1
    if "PARQUET_PATH" in globals() and isinstance(globals()["PARQUET_PATH"], str):
        df_try = _read_any_table(globals()["PARQUET_PATH"])
        px_try = _to_close_wide(df_try)
        if px_try is not None and px_try.shape[1] > 0:
            print("Fuente seleccionada: PARQUET_PATH")
            return px_try, "PARQUET_PATH"

    # 3) Fallback por escaneo de ficheros
    candidates = _scan_candidate_files()

    if len(candidates) == 0:
        raise FileNotFoundError(
            "No se encontraron fuentes validas. Ejecuta NB2 en el mismo kernel o verifica que exista sp500_history.parquet."
        )

    best_px = None
    best_path = None
    best_score = -1

    for path in candidates:
        df = _read_any_table(path)
        px = _to_close_wide(df)

        if px is None:
            continue

        score = px.shape[0] * px.shape[1]

        if px.shape[1] >= min_assets and px.shape[0] >= 260:
            print(f"Fuente seleccionada: {path}")
            return px, path

        if score > best_score:
            best_score = score
            best_px = px
            best_path = path

    if best_px is not None:
        print(f"Fuente seleccionada (mejor disponible): {best_path}")
        return best_px, best_path

    raise ValueError(
        "No fue posible construir una matriz diaria de close. Ejecuta NB2 en el mismo kernel o verifica archivos de salida."
    )


px_close_daily, source_path = load_prepared_close_prices(min_assets=20)

print("Shape diario (close):", px_close_daily.shape)
print("Rango diario:", px_close_daily.index.min(), "->", px_close_daily.index.max())
print("Numero de activos diarios:", px_close_daily.shape[1])
print("Fuente usada:", source_path)


Fuente seleccionada: C:\\Users\\alons\\Desktop\\Práctica 7\\sp500_history.parquet
Shape diario (close): (9087, 1289)
Rango diario: 1990-01-02 00:00:00 -> 2026-01-30 00:00:00
Numero de activos diarios: 1289
Fuente usada: C:\\Users\\alons\\Desktop\\Práctica 7\\sp500_history.parquet


## Universo final: activos vivos (sin GLD)

Regla:
- Se toma universo de activos con datos en los ultimos 13 meses del panel mensual.
- GLD queda explicitamente fuera del universo.
- Si el universo final tiene menos de 20 activos, se lanza error claro.


In [4]:
# Construccion de Pm mensual y universo limpio (sin GLD, sin sesgo futuro)

# 1) Intentamos tomar precios desde variables ya cargadas en memoria
if "Pm" in globals() and isinstance(Pm, pd.DataFrame):
    prices_input = Pm.copy()
elif "px_close_daily" in globals() and isinstance(px_close_daily, pd.DataFrame):
    prices_input = px_close_daily.copy()
elif "datos_analisis_df" in globals() and isinstance(datos_analisis_df, pd.DataFrame):
    prices_input = datos_analisis_df.copy()
elif "datos_base_df" in globals() and isinstance(datos_base_df, pd.DataFrame):
    prices_input = datos_base_df.copy()
elif "df_prepared" in globals() and isinstance(df_prepared, pd.DataFrame):
    prices_input = df_prepared.copy()
else:
    px_close_daily, source_path = load_prepared_close_prices(min_assets=20)
    prices_input = px_close_daily.copy()

# 2) Si viene en formato long y trae bandera sp500, filtramos ex-ante
if isinstance(prices_input, pd.DataFrame):
    col_map = {str(c).strip().lower(): c for c in prices_input.columns}
    date_col = next((col_map[k] for k in ["date", "datetime", "timestamp", "fecha"] if k in col_map), None)
    ticker_col = next((col_map[k] for k in ["ticker", "symbol", "asset", "activo"] if k in col_map), None)
    close_col = next((col_map[k] for k in ["adj_close", "close", "unadjusted_close", "price"] if k in col_map), None)
    flag_col = next((col_map[k] for k in ["in_sp500", "is_sp500", "sp500_flag"] if k in col_map), None)

    if date_col is not None and ticker_col is not None and close_col is not None and flag_col is not None:
        tmp = prices_input.copy()
        flag = tmp[flag_col]
        if str(flag.dtype).lower() == "bool":
            mask = flag.fillna(False)
        else:
            mask = pd.to_numeric(flag, errors="coerce").fillna(0) > 0

        tmp = tmp[mask].copy()
        if len(tmp) > 0:
            prices_input = tmp
            print("Filtro in_sp500 aplicado antes de construir matriz de precios.")

px_close = to_wide_close(prices_input)
px_close = px_close.sort_index().replace([np.inf, -np.inf], np.nan)
Pm_full = to_monthly_last(px_close)

# 3) Warm-up: mantenemos solo desde 13 meses antes del inicio del backtest
Pm_full = Pm_full[Pm_full.index >= WARMUP_START].copy()
px_close = px_close[px_close.index >= WARMUP_START].copy()

if Pm_full.shape[0] < 14:
    raise ValueError("No hay suficientes meses de historia para calcular R12 con lag (se requieren al menos 14 meses).")

# 4) Limpieza de tickers: fuera GLD, fuera sufijo Q de 5 letras, formato ticker razonable
Pm_full = Pm_full.drop(columns=["GLD"], errors="ignore")

tickers = pd.Index([str(c).strip().upper() for c in Pm_full.columns])
valid_format = tickers.to_series(index=tickers).str.fullmatch(r"[A-Z]{1,5}(\.[A-Z])?").fillna(False)
ticker_len = tickers.to_series(index=tickers).str.len().fillna(0)
otc_q_like = tickers.to_series(index=tickers).str.endswith("Q") & (ticker_len >= 5)

clean_tickers = sorted(set(tickers[valid_format & (~otc_q_like)].tolist()))
Pm_full = Pm_full.reindex(columns=clean_tickers)
px_close = px_close.reindex(columns=clean_tickers)

# 5) Intentamos intersecar con universo S&P desde raw parquet (si existe flag)
def _load_sp500_set_from_raw():
    candidate_paths = []
    if "PARQUET_PATH" in globals() and isinstance(PARQUET_PATH, str):
        candidate_paths.append(PARQUET_PATH)
    if "source_path" in globals() and isinstance(source_path, str):
        candidate_paths.append(source_path)

    known = [
        r"C:\Users\alons\Desktop\Pr?ctica 7\sp500_history.parquet",
        r"C:\Users\alons\Desktop\Practica 7\sp500_history.parquet",
        r"data\raw\sp500_history.parquet",
        r"..\data\raw\sp500_history.parquet",
    ]
    for k in known:
        if k not in candidate_paths:
            candidate_paths.append(k)

    local_fs = fs.LocalFileSystem()

    for path in candidate_paths:
        try:
            info = local_fs.get_file_info(path)
            if info.type != fs.FileType.File:
                continue

            table = pq.read_table(path)
            cols_lower = [str(c).strip().lower() for c in table.column_names]

            dcol = next((c for c in ["date", "datetime", "timestamp", "fecha"] if c in cols_lower), None)
            tcol = next((c for c in ["symbol", "ticker", "asset", "activo"] if c in cols_lower), None)
            fcol = next((c for c in ["in_sp500", "is_sp500", "sp500_flag"] if c in cols_lower), None)

            if dcol is None or tcol is None or fcol is None:
                continue

            use_cols = [dcol, tcol, fcol]
            t2 = pq.read_table(path, columns=use_cols)
            raw = t2.to_pandas()

            raw.columns = [str(c).strip().lower() for c in raw.columns]
            raw[dcol] = pd.to_datetime(raw[dcol], errors="coerce")
            raw = raw.dropna(subset=[dcol, tcol])
            raw = raw[(raw[dcol] >= WARMUP_START) & (raw[dcol] <= Pm_full.index.max())]

            if len(raw) == 0:
                continue

            flag = raw[fcol]
            if str(flag.dtype).lower() == "bool":
                mask = flag.fillna(False)
            else:
                mask = pd.to_numeric(flag, errors="coerce").fillna(0) > 0

            tick_set = set(raw.loc[mask, tcol].astype(str).str.upper().unique().tolist())
            if len(tick_set) > 0:
                return tick_set, path
        except Exception:
            continue

    return set(), None

sp500_set, sp500_source = _load_sp500_set_from_raw()
if len(sp500_set) > 0:
    inter = sorted(set(Pm_full.columns).intersection(sp500_set))
    if len(inter) >= TOP_N:
        Pm_full = Pm_full.reindex(columns=inter)
        px_close = px_close.reindex(columns=inter)
        print(f"Universo intersectado con in_sp500 desde: {sp500_source} -> {len(inter)} tickers")
    else:
        print("Aviso: interseccion in_sp500 deja menos de TOP_N; no se aplica para evitar romper notebook.")

# 6) Sin sesgo futuro: exigimos historial minimo pre-backtest (13 observaciones mensuales)
pre_start_monthly = Pm_full[Pm_full.index < BACKTEST_START]
if pre_start_monthly.shape[0] < 13:
    raise ValueError(
        f"Warm-up insuficiente: solo {pre_start_monthly.shape[0]} meses antes del backtest; se requieren al menos 13."
    )

has_hist_13m = pre_start_monthly.notna().sum(axis=0) >= 13

# 7) Filtro anti-outliers usando datos diarios pre-backtest
pre_start_daily = px_close[(px_close.index < BACKTEST_START) & (px_close.index >= WARMUP_START)].copy()
obs_ok = pre_start_daily.notna().sum(axis=0) >= MIN_DAILY_OBS_PRE

if pre_start_daily.shape[0] == 0:
    raise ValueError("No hay datos diarios en ventana pre-backtest para filtros anti-outliers.")

last_pre_price = pre_start_daily.ffill().iloc[-1]
price_ok = last_pre_price >= MIN_LAST_PRICE_PRE

ret_pre = pre_start_daily.pct_change().replace([np.inf, -np.inf], np.nan)
p99_abs = ret_pre.abs().quantile(0.99)
max_abs = ret_pre.abs().max()
vol_ann = ret_pre.std(skipna=True) * np.sqrt(252)

stable_ret_ok = p99_abs <= MAX_P99_ABS_RET_PRE
max_jump_ok = max_abs <= MAX_MAX_ABS_RET_PRE
vol_ok = vol_ann <= MAX_ANNUAL_VOL_PRE

quality_mask = has_hist_13m & obs_ok & price_ok & stable_ret_ok & max_jump_ok & vol_ok

universe_tickers = sorted(set(quality_mask[quality_mask].index.tolist()))
removed_tickers = sorted(set(Pm_full.columns) - set(universe_tickers))

Pm = Pm_full[universe_tickers].copy()

if Pm.shape[1] < TOP_N:
    raise ValueError(
        f"Universo insuficiente tras limpieza/outliers: {Pm.shape[1]} activos validos. Se requieren al menos {TOP_N}."
    )

print("Rango mensual (con warm-up):", Pm.index.min(), "->", Pm.index.max())
print("Meses pre-backtest:", pre_start_monthly.shape[0])
print("Numero de activos en universo final limpio (sin GLD):", Pm.shape[1])
print("Activos eliminados por filtros anti-outliers:", len(removed_tickers))
if len(removed_tickers) > 0:
    print("Ejemplo eliminados:", removed_tickers[:25])


C:\Users\alons\AppData\Local\Temp\ipykernel_3136\3254323303.py:67: FutureWarning: 'BM' is deprecated and will be removed in a future version, please use 'BME' instead.
  Pm = px.resample("BM").last()


Universo intersectado con in_sp500 desde: C:\\Users\\alons\\Desktop\\Práctica 7\\sp500_history.parquet -> 603 tickers
Rango mensual (con warm-up): 2013-12-31 00:00:00 -> 2026-01-30 00:00:00
Meses pre-backtest: 13
Numero de activos en universo final limpio (sin GLD): 532
Activos eliminados por filtros anti-outliers: 71
Ejemplo eliminados: ['ABNB', 'AIV', 'AMCR', 'AMD', 'AMTM', 'ANET', 'APP', 'ARES', 'BHF', 'CARR', 'CEG', 'CFG', 'COIN', 'CPRT', 'CRH', 'CRWD', 'CTVA', 'CVNA', 'CZR', 'DASH', 'DAY', 'DD', 'DDOG', 'DELL', 'DOW']


C:\Users\alons\AppData\Local\Temp\ipykernel_3136\1527316635.py:153: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret_pre = pre_start_daily.pct_change().replace([np.inf, -np.inf], np.nan)


## Paso A/B/C: Momentum, Z-score y Seleccion TOP 20 (sin GLD)

Paso A:
- `R12 = log(Pm.shift(1)/Pm.shift(13))`
- `R6  = log(Pm.shift(1)/Pm.shift(7))`

Paso B:
- Z-score cross-sectional por mes (`Z12`, `Z6`).

Paso C:
- `score = 0.5*(Z12 + Z6)`
- Ranking descendente y TOP 20 por fecha de rebalanceo.
- Si en un mes hay <20 scores validos: mantener pesos del mes anterior (hold-last-weights).


In [5]:
top_n = TOP_N

R12, R6 = compute_momentum(Pm)

# Primera fecha valida basada en SCORES validos (>= top_n) y >= inicio backtest
Z12_tmp = cross_sectional_zscore(R12)
Z6_tmp = cross_sectional_zscore(R6)
score_tmp = 0.5 * (Z12_tmp + Z6_tmp)

valid_score_count = score_tmp.notna().sum(axis=1)
first_rebalance = valid_score_count[(valid_score_count >= top_n) & (valid_score_count.index >= BACKTEST_START)].index.min()

if pd.isna(first_rebalance):
    raise ValueError(
        f"No existe ninguna fecha >= {BACKTEST_START.date()} con al menos {top_n} scores validos."
    )

print("Primera fecha de rebalanceo ejecutable (SCORE>=20 y >= inicio):", first_rebalance)
print("Resumen de activos con score valido por fecha:")
print(valid_score_count.describe())

selection_wide, selection_long = build_scores_and_select(
    R12,
    R6,
    top_n=top_n,
    hold_last_weights=True,
)

# Export/ejecucion solo desde inicio de backtest (warm-up solo para calculo de senales)
selection_wide = selection_wide[selection_wide.index >= BACKTEST_START].copy()
selection_long = selection_long[selection_long["rebalance_date"] >= BACKTEST_START].copy()

# Garantiza maximo TOP_N por fecha en salida
selection_long = (
    selection_long.sort_values(["rebalance_date", "rank", "ticker"], ascending=[True, True, True])
    .groupby("rebalance_date", as_index=False, group_keys=False)
    .head(top_n)
    .reset_index(drop=True)
)

print("\nSanity checks Pm:")
print("Pm shape:", Pm.shape)
print("Pm rango:", Pm.index.min(), "->", Pm.index.max())
print("Pm infer_freq:", pd.infer_freq(Pm.index))
print("Ejemplo fechas Pm:", list(Pm.index[:6]))

print("\nRango final exportado (selection_long):",
      selection_long["rebalance_date"].min(), "->", selection_long["rebalance_date"].max())
print("Fechas rebalance exportadas:", selection_long["rebalance_date"].nunique())

print("\nPreview selection_long:")
display(selection_long.head(30))

print("Shape selection_long:", selection_long.shape)
print("Shape selection_wide:", selection_wide.shape)


Primera fecha de rebalanceo ejecutable (SCORE>=20 y >= inicio): 2015-01-30 00:00:00
Resumen de activos con score valido por fecha:
count    146.000000
mean     484.630137
std      152.036793
min        0.000000
25%      532.000000
50%      532.000000
75%      532.000000
max      532.000000
dtype: float64

Sanity checks Pm:
Pm shape: (146, 532)
Pm rango: 2013-12-31 00:00:00 -> 2026-01-30 00:00:00
Pm infer_freq: BME
Ejemplo fechas Pm: [Timestamp('2013-12-31 00:00:00'), Timestamp('2014-01-31 00:00:00'), Timestamp('2014-02-28 00:00:00'), Timestamp('2014-03-31 00:00:00'), Timestamp('2014-04-30 00:00:00'), Timestamp('2014-05-30 00:00:00')]

Rango final exportado (selection_long): 2015-01-30 00:00:00 -> 2026-01-30 00:00:00
Fechas rebalance exportadas: 133

Preview selection_long:


,rebalance_date,ticker,rank,score,z12,z6,r12,r6,weight,rebalanced
0,2015-01-30,SWKS,1,3.043440,3.864203,2.222677,0.941386,0.441439,0.05,1
1,2015-01-30,ENPH,2,2.943820,3.241953,2.645687,0.812681,0.513629,0.05,1
2,2015-01-30,LUV,3,2.790580,3.261468,2.319692,0.816718,0.457995,0.05,1
3,2015-01-30,AXON,4,2.728061,1.785004,3.671118,0.511329,0.688626,0.05,1
4,2015-01-30,PANW,5,2.417703,2.974777,1.860628,0.757419,0.379653,0.05,1
5,2015-01-30,UAL,6,2.281288,2.068404,2.494172,0.569947,0.487772,0.05,1
6,2015-01-30,EW,7,2.229072,2.509406,1.948738,0.661163,0.394690,0.05,1
7,2015-01-30,AVGO,8,2.064740,2.497317,1.632163,0.658662,0.340664,0.05,1
8,2015-01-30,RCL,9,2.031350,2.071916,1.990783,0.570673,0.401865,0.05,1
9,2015-01-30,EA,10,2.001798,2.782188,1.221408,0.717585,0.270565,0.05,1


Shape selection_long: (2660, 10)
Shape selection_wide: (133, 532)


## Export CSV obligatorio

Se guarda `selection_long` en:
- `outputs/selected_top20_by_rebalance.csv`

Si falla, fallback a:
- `./selected_top20_by_rebalance.csv`


In [6]:
save_primary = "outputs/selected_top20_by_rebalance.csv"
save_fallback = "selected_top20_by_rebalance.csv"
saved_path = None

try:
    fs.LocalFileSystem().create_dir("outputs", recursive=True)
    selection_long.to_csv(save_primary, index=False)
    saved_path = save_primary
except Exception as e1:
    print("No se pudo guardar en outputs/:", str(e1))

if saved_path is None:
    try:
        selection_long.to_csv(save_fallback, index=False)
        saved_path = save_fallback
    except Exception as e2:
        raise RuntimeError(f"No se pudo guardar el CSV ni en outputs/ ni en raiz. Detalle: {e2}")

print("CSV guardado en:", saved_path)
print("Filas guardadas:", len(selection_long))
if len(selection_long) > 0:
    print("Rango de fechas guardado:", selection_long["rebalance_date"].min(), "->", selection_long["rebalance_date"].max())


CSV guardado en: outputs/selected_top20_by_rebalance.csv
Filas guardadas: 2660
Rango de fechas guardado: 2015-01-30 00:00:00 -> 2026-01-30 00:00:00


## Resultado final para Notebook 4

Variables disponibles:
- `Pm`: precios mensuales del universo final.
- `R12`, `R6`: momentum logaritmico con lag de 1 mes.
- `selection_wide`: tickers seleccionados por rank para cada rebalanceo.
- `selection_long`: trazabilidad completa de score y factores.

En este notebook NO se implementan costes ni ejecucion de ordenes (eso corresponde a Notebook 4).
